# 🛒 Online Retail Store — Exploratory Data Analysis (EDA)

**Dataset:** UCI Online Retail Dataset  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn  
**Author:** Saurabh  
**Objective:** Analyse an online retail dataset to uncover sales trends, customer behaviour patterns, and product performance insights that support data-driven business decisions.

---

## 📌 Table of Contents
1. Import Libraries
2. Load Dataset
3. Data Understanding
4. Data Cleaning
5. Statistical Analysis
6. Data Visualisation
7. Correlation Heatmap
8. Key Findings & Business Recommendations

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style for professional-looking charts
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

## 2. Load Dataset

In [ ]:
df = pd.read_excel('online retail.xlsx')
df.head(10)

## 3. Data Understanding

Understanding the structure, shape, and data types of the dataset before any analysis.

In [ ]:
# Shape of the dataset
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

In [ ]:
# Column names
print("Columns:", df.columns.tolist())

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Check missing values
missing = df.isnull().sum()
print("Missing Values:")
print(missing[missing > 0])

## 4. Data Cleaning

Cleaning the data to ensure accuracy and reliability before analysis.  
Steps: handle missing values → remove duplicates → filter cancelled orders → remove invalid quantities/prices → feature engineering.

In [ ]:
# Step 1: Drop rows with missing CustomerID (required for customer analysis)
print(f"Rows before dropping missing CustomerID: {len(df)}")
df = df.dropna(subset=['CustomerID'])
print(f"Rows after dropping missing CustomerID: {len(df)}")

In [ ]:
# Step 2: Remove duplicate rows
print(f"Rows before removing duplicates: {len(df)}")
df = df.drop_duplicates()
print(f"Rows after removing duplicates: {len(df)}")

In [ ]:
# Step 3: Identify cancelled orders (InvoiceNo starting with 'C')
cancelled = df[df['InvoiceNo'].astype(str).str.startswith('C')]
print(f"Cancelled orders found: {len(cancelled)}")
print(cancelled[['InvoiceNo', 'Quantity', 'Description']].head(5))

In [ ]:
# Remove cancelled orders from the dataset
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
print(f"Rows after removing cancelled orders: {len(df)}")

In [ ]:
# Step 4: Remove rows with negative or zero Quantity and UnitPrice
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
print(f"Rows after removing invalid Quantity/UnitPrice: {len(df)}")

In [ ]:
# Step 5: Feature Engineering — create Revenue/Sales column
df['Sales'] = df['Quantity'] * df['UnitPrice']

# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Extract Month and Day for time-based analysis
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

df['Month'] = df['InvoiceDate'].dt.month_name()
df['Day'] = df['InvoiceDate'].dt.day_name()

print("New columns added: Sales, Month, Day")
df[['InvoiceDate', 'Sales', 'Month', 'Day']].head(5)

## 5. Statistical Analysis

Descriptive statistics to understand the distribution of key numerical variables.

In [ ]:
df[['Quantity', 'UnitPrice', 'Sales']].describe().round(2)

In [ ]:
# Total revenue in the dataset
total_revenue = df['Sales'].sum()
total_orders = df['InvoiceNo'].nunique()
total_customers = df['CustomerID'].nunique()
total_products = df['Description'].nunique()
total_countries = df['Country'].nunique()

print(f"💰 Total Revenue     : £{total_revenue:,.2f}")
print(f"📦 Total Orders      : {total_orders:,}")
print(f"👥 Unique Customers  : {total_customers:,}")
print(f"🏷️  Unique Products   : {total_products:,}")
print(f"🌍 Countries Served  : {total_countries:,}")

## 6. Data Visualisation

### 6.1 Sales Distribution

In [ ]:
# Capping at £500 to show the true distribution (extreme outliers compress the chart)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df[df['Sales'] < 500]['Sales'], bins=60, color='steelblue', kde=True, ax=axes[0])
axes[0].set_title('Sales Distribution (Capped at £500)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sales (£)')
axes[0].set_ylabel('Number of Transactions')

sns.histplot(df[df['Quantity'] < 100]['Quantity'], bins=50, color='coral', kde=True, ax=axes[1])
axes[1].set_title('Quantity Distribution (Capped at 100)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Quantity')
axes[1].set_ylabel('Number of Transactions')

plt.tight_layout()
plt.show()

> **Insight:** Most transactions are small-value (under £50), indicating a high volume of low-ticket purchases. A small number of bulk orders drive the upper tail.

### 6.2 Quantity vs Sales

In [ ]:
# Filter out extreme outliers for a cleaner scatter plot
df_plot = df[(df['Quantity'] < 200) & (df['Sales'] < 2000)]

plt.figure(figsize=(9, 5))
sns.scatterplot(x='Quantity', y='Sales', data=df_plot, alpha=0.3, color='teal')
plt.title('Quantity vs Sales', fontsize=13, fontweight='bold')
plt.xlabel('Quantity Ordered')
plt.ylabel('Sales Revenue (£)')
plt.tight_layout()
plt.show()

> **Insight:** A clear positive relationship — as quantity increases, revenue increases. The spread suggests varying unit prices across products.

### 6.3 Monthly Sales Trend

In [ ]:
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']

monthly_sales = df.groupby('Month')['Sales'].sum().reindex(month_order)

plt.figure(figsize=(12, 6))
bars = plt.bar(monthly_sales.index, monthly_sales.values, color='steelblue', edgecolor='white')

# Highlight the peak month
peak_month = monthly_sales.idxmax()
for bar, month in zip(bars, monthly_sales.index):
    if month == peak_month:
        bar.set_color('coral')

plt.title('Monthly Sales Trend', fontsize=14, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Total Revenue (£)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"\n📈 Peak Sales Month: {peak_month} — £{monthly_sales[peak_month]:,.2f}")

> **Insight:** Sales peak sharply in **November**, driven by pre-Christmas shopping. Revenue drops significantly in January and February — typical post-holiday slowdown. The business should plan inventory and marketing campaigns around Q4.

### 6.4 Sales by Day of Week

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

daily_sales = df.groupby('Day')['Sales'].sum().reindex(day_order)

plt.figure(figsize=(10, 5))
daily_sales.plot(kind='bar', color='mediumseagreen', edgecolor='white')
plt.title('Revenue by Day of Week', fontsize=14, fontweight='bold')
plt.xlabel('Day of Week', fontsize=12)
plt.ylabel('Total Revenue (£)', fontsize=12)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(f"\n📅 Highest Revenue Day: {daily_sales.idxmax()} — £{daily_sales.max():,.2f}")

> **Insight:** Thursday and Tuesday are the highest revenue days. Sunday shows near-zero sales — likely because this is a B2B-oriented retailer that doesn't trade over weekends.

### 6.5 Top 10 Best-Selling Products

In [ ]:
top_products = df.groupby('Description')['Quantity'].sum().sort_values(ascending=True).tail(10)

plt.figure(figsize=(10, 6))
top_products.plot(kind='barh', color='slateblue', edgecolor='white')
plt.title('Top 10 Best-Selling Products by Quantity', fontsize=13, fontweight='bold')
plt.xlabel('Total Quantity Sold', fontsize=12)
plt.ylabel('')
plt.tight_layout()
plt.show()

> **Insight:** The top products are primarily small decorative and gift items — consistent with the retailer's profile as a UK-based giftware wholesaler. These items should always be kept well-stocked.

### 6.6 Top Countries by Revenue

In [ ]:
country_sales = df.groupby('Country')['Sales'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
country_sales.plot(kind='bar', color='darkorange', edgecolor='white')
plt.title('Top 10 Countries by Revenue', fontsize=14, fontweight='bold')
plt.xlabel('Country', fontsize=12)
plt.ylabel('Total Revenue (£)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

> **Insight:** The UK dominates revenue — expected since this is a UK-based retailer. Among international markets, **Netherlands, EIRE (Ireland), and Germany** are the strongest, indicating good export potential to Europe.

In [ ]:
# Exclude UK to better see international performance
country_sales_intl = df[df['Country'] != 'United Kingdom'].groupby('Country')['Sales'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
country_sales_intl.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Top 10 International Markets (Excl. UK)', fontsize=13, fontweight='bold')
plt.xlabel('Country', fontsize=12)
plt.ylabel('Total Revenue (£)', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

> **Insight:** Netherlands and EIRE lead international sales. Expanding marketing efforts in these regions could significantly grow cross-border revenue.

### 6.7 Top 10 Customers by Revenue

In [ ]:
top_customers = df.groupby('CustomerID')['Sales'].sum().sort_values(ascending=False).head(10).reset_index()
top_customers['CustomerID'] = top_customers['CustomerID'].astype(int).astype(str)

plt.figure(figsize=(12, 6))
plt.bar(top_customers['CustomerID'], top_customers['Sales'], color='mediumpurple', edgecolor='white')
plt.title('Top 10 Customers by Revenue', fontsize=14, fontweight='bold')
plt.xlabel('Customer ID', fontsize=12)
plt.ylabel('Total Spending (£)', fontsize=12)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(f"\n🏆 Top Customer Revenue: £{top_customers['Sales'].iloc[0]:,.2f}")

> **Insight:** The top customer alone contributes significantly more than others — a classic **Pareto (80/20) pattern**. A loyalty or retention programme targeting these high-value customers would protect a large portion of revenue.

### 6.8 Outlier Detection

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x=df['Quantity'], color='lightcoral', ax=axes[0])
axes[0].set_title('Outliers in Quantity', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Quantity')

sns.boxplot(x=df['UnitPrice'], color='lightblue', ax=axes[1])
axes[1].set_title('Outliers in Unit Price', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Unit Price (£)')

plt.tight_layout()
plt.show()

> **Insight:** Both Quantity and UnitPrice have significant outliers. These likely represent **bulk wholesale orders** rather than errors — a characteristic of B2B retail. These records were retained since they represent real business transactions.

## 7. Correlation Heatmap

A correlation heatmap shows how strongly numerical variables are related to each other.  
Values range from **-1** (opposite relationship) to **+1** (perfect relationship). **0** means no relationship.

In [ ]:
plt.figure(figsize=(7, 5))
correlation = df[['Quantity', 'UnitPrice', 'Sales']].corr()

sns.heatmap(correlation, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation Between Key Variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(correlation.round(2))

> **Insight:**  
> - **Sales vs Quantity (strong positive):** Expected — Sales = Quantity × Price, so they naturally move together.  
> - **UnitPrice vs Quantity (weak/negative):** Customers don't buy significantly more of an item just because it's cheaper — consistent with B2B bulk purchasing behaviour where order quantity is driven by business need, not price sensitivity.

## 8. Key Findings & Business Recommendations

---

### 📊 Key Findings

| # | Finding |
|---|---------|
| 1 | **November is the peak sales month** — driven by pre-Christmas demand |
| 2 | **Thursday and Tuesday** generate the highest weekday revenue |
| 3 | **Sunday sales are near zero** — confirming a B2B wholesale business model |
| 4 | **UK dominates revenue** but Netherlands, EIRE, and Germany are strong international markets |
| 5 | **Top 10 customers** contribute a disproportionately large share of total revenue (Pareto principle) |
| 6 | **Small decorative and giftware items** are the best-selling products by volume |
| 7 | **Outliers in Quantity** represent genuine bulk orders, not data errors |

---

### 💡 Business Recommendations

1. **Seasonal Strategy:** Launch Q4 promotions and increase inventory from October to capture peak demand in November.
2. **Customer Retention:** Identify and create a loyalty programme for the top 20% of customers who drive 80% of revenue.
3. **International Expansion:** Invest in targeted marketing for Netherlands, Germany, and EIRE — the strongest export markets.
4. **Weekend Engagement:** Explore B2C channels or weekend promotions since Sunday revenue is virtually zero.
5. **Pricing Analysis:** Since UnitPrice has little impact on Quantity ordered, value-based pricing strategies can be tested without risking volume loss.

---

*Analysis by Saurabh | Tools: Python, Pandas, Matplotlib, Seaborn*